# Layerwise Jacobian singular-value distribution for heavy-tailed MLPs

Visualisation of `ht_mlp_jacobian.py` / `ht_mlp_jacobian.md`. Four pathways target the SV density of $J^l = D^l W^l$ at the heavy-tailed MFT fixed point $q^*$:

* **(P1)** Theorem-2 deterministic-quantile theory -- single scalar closure in $Y_r$ (`RMT/one_sided_wishart_levy.py`; dedicated Theorem 2 solver with anchor-first seed continuation, imag-eps homotopy, and *independent*-rule $h_\alpha$ density readout for accuracy at small $s$).
* **(P2)** Population-dynamics cavity-equation route (`RMT.py:jac_cavity_svd_log_pdf`).
* **(P3a)** Synthetic empirical SVDs: $h_j \sim (q^*)^{1/\alpha} p_\alpha$ directly, $J = D W$, batch SVDs.
* **(P3b)** MLP-derived empirical SVDs: `RMT.MLP` at depth where $q^l \to q^*$.

Each cell times its body; results saved to `fig/ht_mlp_jacobian/` for the panel sweep.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

cwd = Path.cwd()
if (cwd / 'ht_mlp_jacobian.py').exists():
    sys.path.insert(0, str(cwd))
elif (cwd / 'random_dnn' / 'ht_mlp_jacobian.py').exists():
    sys.path.insert(0, str(cwd / 'random_dnn'))

import ht_mlp_jacobian as htj
sys.path.insert(0, str(cwd / 'RMT') if (cwd / 'RMT').is_dir() else str(cwd / 'random_dnn' / 'RMT'))
import one_sided_wishart_levy as osw
import localisation as loc

FIGDIR = cwd / 'fig' / 'ht_mlp_jacobian'
FIGDIR.mkdir(parents=True, exist_ok=True)

## 0. Convention check

Sanity gate: the structured-curve native sampler `swl.sample_structured_levy_matrix(entry_scale=sigma_w)` should produce the same SV histogram as the MLP-style sampler `sigma_w * (2N)**(-1/alpha) * scipy_stable(scale=1)` (Belinschi-vs-SciPy factor absorbed into the prefactor). Pass criterion: agreement within Monte-Carlo bin noise.

In [ ]:
conv = htj.convention_check(alpha=1.5, sigma_w=1.0, N=256, num_matrices=80, bins=121, seed=0)
print(f"max abs density diff: {conv['max_abs_diff']:.4f}")
print(f"relative L1 diff:     {conv['rel_L1_diff']:.4f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(conv['centres'], conv['density_structured_native'], label='structured native')
ax.plot(conv['centres'], conv['density_mlp_style'], '--', label='MLP-style')
ax.set(xlabel='singular value $s$', ylabel='density', title=r'convention check ($\alpha=1.5$, $\sigma_w=1$)')
ax.legend(); fig.tight_layout()
fig.savefig(FIGDIR / 'convention_check.png', dpi=120)
plt.show()

## 1. Three-way validation at a representative point

$(\alpha, \sigma_w) = (1.5, 1.0)$, $\phi = \tanh$. The forward MFT gives $q^* \approx 0.106$, $(q^*)^{1/\alpha} \approx 0.224$.

All four pathways (P1, P2, P3a, P3b) plotted on the same SV axis.

In [ ]:
out = htj.run_validation(
    alpha=1.5, sigma_W=1.0, N=256, num_matrices=40,
    depth=50, burn_in=25, num_doublings=7, num_chis=1,
    s_max=6.0, num_points=121, bins=61,
    n_profile_samples=30_000, do_mlp=True, seed=0,
    save_path=str(FIGDIR / 'validation_alpha1.5_sigma1.0.npz'),
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Linear-scale density comparison
ax1.plot(out['sv_theory'][1:], out['density_theory'][1:],
         color='C0', lw=2, label='(P1) Theorem 2 theory')
ax1.plot(out['sv_grid_pop_dyn'], out['density_pop_dyn'],
         color='C1', lw=1, label='(P2) population dynamics')
ax1.plot(out['sv_synthetic_centres'], out['density_synthetic'],
         'o', color='C2', ms=4, label='(P3a) synthetic empirical')
if len(out['sv_mlp_centres']):
    ax1.plot(out['sv_mlp_centres'], out['density_mlp'],
             's', color='C3', ms=4, mfc='none', label='(P3b) MLP empirical')
ax1.set(xlabel='singular value $s$', ylabel='density',
        title=fr"$\alpha=1.5$, $\sigma_w=1.0$, $q^*={out['summary']['q_star']:.3f}$")
ax1.legend(); ax1.set_xlim(0, 4)

# Tail: log-log with all pathways + B s^{-(1+alpha)} reference
B = out['summary']['tail_constant_B']
s_tail = np.linspace(2.0, 8.0, 50)
ax2.loglog(out['sv_theory'][1:], out['density_theory'][1:],
           color='C0', lw=2, label='(P1) theory')
ax2.loglog(out['sv_grid_pop_dyn'], out['density_pop_dyn'],
           color='C1', lw=1, label='(P2) population dynamics')
ax2.loglog(out['sv_synthetic_centres'], out['density_synthetic'],
           'o', color='C2', ms=4, label='(P3a) synthetic')
if len(out['sv_mlp_centres']):
    ax2.loglog(out['sv_mlp_centres'], out['density_mlp'],
               's', color='C3', ms=4, mfc='none', label='(P3b) MLP empirical')
ax2.loglog(s_tail, B * s_tail ** (-2.5),
           ':', color='k', label=fr'$B\,s^{{-(1+\alpha)}}$, $B={B:.3f}$')
ax2.set(xlabel='singular value $s$', ylabel='density',
        title='heavy tail $f(s) \\sim B\\,s^{-(1+\\alpha)}$',
        ylim=(1e-3, 3.0))
ax2.legend()

fig.tight_layout()
fig.savefig(FIGDIR / 'validation_alpha1.5_sigma1.0.png', dpi=120)
plt.show()
print('summary diffs:')
for k in ('diff_theory_synthetic', 'diff_pop_dyn_synthetic', 'diff_theory_pop_dyn',
         'convention_max_abs_diff', 'q_rel_err_mlp'):
    print(f'  {k}: {out["summary"][k]:.4f}')

## 2. Panel across $(\alpha, \sigma_w)$

The shape evolves as $\alpha$ moves from heavy-tailed ($\alpha=1.3$, fat tail / sharp peak at small $s$) toward Gaussian ($\alpha=1.9$, Marchenko-Pastur-like). $\sigma_w$ controls $q^*$ and hence the column profile $c$.

Each panel: (P1) theory + (P3a) synthetic empirical. The pop-dynamics and MLP-derived routes are omitted for compactness but verified at the representative point above.

In [ ]:
alphas = [1.3, 1.5, 1.7]
sigmas = [0.8, 1.2, 1.8]
fig, axes = plt.subplots(len(alphas), len(sigmas), figsize=(4 * len(sigmas), 2.6 * len(alphas)), sharex=True)
for i, alpha in enumerate(alphas):
    for j, sw in enumerate(sigmas):
        ax = axes[i, j]
        profile = htj.jacobian_profile(alpha, sw, n_samples=15_000, seed=0)
        curve, _ = htj.theoretical_jacobian_sv_curve(
            alpha, sw, s_max=8.0, num_points=61, profile=profile,
            quadrature_order=64, profile_order=48,
        )
        emp = htj.synthetic_jacobian_sv_spectrum(
            alpha, sw, q_star=profile.q_star,
            N=150, num_matrices=15, seed=0, bins=41,
            sv_range=(0.0, 8.0),
        )
        ax.plot(curve.singular_values[1:], curve.singular_density[1:], 'C0', lw=1.6, label='theory')
        ax.plot(emp['centres'], emp['density'], 'o', color='C2', ms=3, label='empirical')
        ax.set_xlim(0, 5)
        ax.set_title(fr"$\alpha={alpha}$, $\sigma_w={sw}$, $q^*={profile.q_star:.2f}$", fontsize=10)
        if i == len(alphas) - 1: ax.set_xlabel(r'$s$')
        if j == 0: ax.set_ylabel('density')
        if i == 0 and j == 0: ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGDIR / 'panel_alpha_sigma.png', dpi=120)
plt.show()

## 3. Column profile $c(v)$ visualisation

The quantile function $c(v) = F^{-1}_{|\phi'((q^*)^{1/\alpha} Z)|}(v)$, $Z \sim p_\alpha$, is the deterministic profile fed to Theorem 2.

For $\phi = \tanh$, $|\phi'(h)| = 1 - \tanh^2(h) \in (0, 1]$, with $|\phi'(0)| = 1$ (linear regime) and $|\phi'(|h| \to \infty)| \to 0$ (saturation). Heavy-tailed $h$ at scale $(q^*)^{1/\alpha}$ has many saturated neurons -> $c(v)$ is small over a wide low-$v$ tail and rises sharply near $v = 1$.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
v_grid = np.linspace(0, 1, 401)
for alpha, sw, color in [(1.3, 1.0, 'C0'), (1.5, 1.0, 'C1'), (1.7, 1.0, 'C2'), (1.9, 1.0, 'C3')]:
    profile = htj.jacobian_profile(alpha, sw, n_samples=50_000, seed=0)
    c = htj.quantile_callable(profile)
    ax.plot(v_grid, c(v_grid), color=color,
            label=fr"$\alpha={alpha}$ ($(q^*)^{{1/\alpha}}={profile.pre_scale:.3f}$)")
ax.set(xlabel=r'$v \in [0,1]$', ylabel=r"$c(v) = F^{-1}_{|\phi'((q^*)^{1/\alpha} Z)|}(v)$",
       title=r'column profile $c(v)$ at $\sigma_w=1$')
ax.legend(); fig.tight_layout()
fig.savefig(FIGDIR / 'column_profile.png', dpi=120)
plt.show()

## 4. Fixed-point phase diagrams and the SV-localisation boundary

Every Jacobian result above is built from the heavy-tailed length-map fixed point
$q^*(\alpha,\sigma_w)$: the cavity/profile multiplier is
$a_i=\sigma_w\,|\phi'((q^*)^{1/\alpha} Z_i)|$ (the entry scale $\sigma_w$ is part
of the Jacobian $J_{ij}=|\phi'(h_i)|\sigma_w N^{-1/\alpha}\tilde W_{ij}$, and the
preactivation stable scale is $(q^*)^{1/\alpha}$). This panel maps three
fixed-point quantities -- $q^*$, the **saturated fraction**
$P(|\phi'((q^*)^{1/\alpha} Z)|<0.05)$ (drives SV localisation), and the **physical
SV-tail constant** $B=\tfrac{\alpha}{2}\sigma_w^\alpha\,\mathbb{E}|\phi'|^\alpha$
(eq. 7) -- over $\alpha\in[1,1.95]$, $\sigma_w\in[0.05,3]$ at spacing $0.05$
($\alpha=2$, $\sigma_w=0$ excluded). (We do not give $(q^*)^{1/\alpha}$ its own
panel -- it is a monotone reparam of $q^*$.)

**The overlaid contour is the SV-localisation onset.** The mobility edge is a
*field* $s_c(\alpha,\sigma_w)$ (the singular value where the cavity $\eta$-scaling
exponent $p$ crosses $1/2$; sec. 6, `jacobian_localization_edge`). It cannot be
drawn on this fine grid (each point is a ~90s cavity run), but the edge enters the
spectrum exactly as the saturated fraction crosses $\approx 0.10$, so the
$\mathrm{sat}=0.10$ contour **is** the onset boundary and is free. Cell 4b shows
the physical edge $s_c$ itself; cell 4c shows the edge-truncated propagation gain
$\chi_1^{<s_c}$.

**Physical units.** The cavity computes the edge in entry-scale-1 units; since the
singular values of $J$ scale linearly in $\sigma_w$, the physical edge is
$s_c=\sigma_w\times(\text{cavity edge})$, a roughly flat $\sim 10$-$12$ band. The
full mean-square gain $\chi_1=\langle s^2\rangle$ **diverges** (heavy SV tail), so
the meaningful gain is the edge-truncated $\chi_1^{<s_c}$ of cell 4c.

In [ ]:
import time as _time
from matplotlib.colors import LogNorm

# Fixed-point grid: alpha 1.00-1.95, sigma_w 0.05-3.00, spacing 0.05
# (alpha=2 and sigma_w=0 excluded).  Each point: one jacobian_profile call.
alphas = np.round(np.arange(1.00, 2.00, 0.05), 2)
sigmas = np.round(np.arange(0.05, 3.0001, 0.05), 2)
nA, nS = len(alphas), len(sigmas)
qstar = np.full((nS, nA), np.nan)
satfr = np.full((nS, nA), np.nan); Malpha = np.full((nS, nA), np.nan)
_t0 = _time.time()
for j, a in enumerate(alphas):
    for i, sw in enumerate(sigmas):
        prof = htj.jacobian_profile(float(a), float(sw), n_samples=20_000, seed=0)
        s = prof.profile_samples
        qstar[i, j] = prof.q_star
        satfr[i, j] = float((s < 0.05).mean()); Malpha[i, j] = prof.profile_alpha_moment
print(f"[grid {nS}x{nA} = {nS*nA} pts] elapsed {_time.time() - _t0:.1f}s")

# Physical SV-density tail constant B = (alpha/2) sigma_w^alpha E|phi'|^alpha (eq. 7).
A = alphas[None, :]; SW = sigmas[:, None]
Bphys = (A / 2.0) * SW ** A * Malpha

ONSET = 0.10  # saturated fraction at which the SV-localisation edge enters
ext = [alphas[0] - 0.025, alphas[-1] + 0.025, sigmas[0] - 0.025, sigmas[-1] + 0.025]


def _panel(ax, Z, title, lognorm=False, cmap="viridis"):
    if lognorm:
        Zp = np.clip(Z, np.nanmax(Z) * 1e-6, None)
        im = ax.imshow(Zp, origin="lower", aspect="auto", extent=ext, cmap=cmap,
                       norm=LogNorm(vmin=np.nanmin(Zp), vmax=np.nanmax(Zp)))
    else:
        im = ax.imshow(Z, origin="lower", aspect="auto", extent=ext, cmap=cmap)
    cs = ax.contour(alphas, sigmas, satfr, levels=[ONSET], colors="white", linewidths=2)
    ax.clabel(cs, fmt={ONSET: "loc. onset"}, fontsize=8)
    ax.set(xlabel=r"$\alpha$", ylabel=r"$\sigma_w$", title=title)
    plt.colorbar(im, ax=ax)


# Three fixed-point quantities (S* dropped -- it is just (q*)^{1/alpha}, a monotone
# reparam of q*).  q* and the saturated fraction are preactivation/profile
# quantities; B is the physical SV-density tail constant.
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
_panel(ax[0], qstar, r"$q^*$ (length-map fixed point)", lognorm=True)
_panel(ax[1], satfr, r"saturated fraction $P(|\phi'((q^*)^{1/\alpha}Z)|<0.05)$", cmap="magma")
_panel(ax[2], Bphys, r"$B=\frac{\alpha}{2}\sigma_w^\alpha\mathbb{E}|\phi'|^\alpha$ (physical SV-tail const.)",
       lognorm=True, cmap="cividis")
fig.suptitle(r"Heavy-tailed MLP fixed-point quantities (physical); white contour = SV-localisation onset (sat$=0.10$)",
             fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(FIGDIR / "fixed_point_phase.png", dpi=120)
plt.show()

i15 = int(np.argmin(np.abs(alphas - 1.5)))
onset_sw = sigmas[np.argmax(satfr[:, i15] > ONSET)]
print(f"sat range {np.nanmin(satfr):.3f}-{np.nanmax(satfr):.3f}; "
      f"localisation onset at alpha=1.5: sigma_w ~ {onset_sw:.2f}; "
      f"B range {np.nanmin(Bphys):.3g}-{np.nanmax(Bphys):.3g}")

### 4b. Physical mobility edge $s_c(\alpha,\sigma_w)$

The mobility-edge field itself, as a grid (cf. the fixed-point panels above).
Values are the **physical** edge $s_c=\sigma_w\times(\text{cavity edge})$ from
`jacobian_localization_phase` (each cell a ~90s cavity run, so the grid is coarse
and precomputed below; uncomment the call to regenerate). `deloc` cells have
$p<\tfrac12$ everywhere (no edge); the region below the grid is delocalised by the
unstructured Bordenave-Guionnet baseline. In physical singular-value units the
edge is a roughly flat $\sim 10$-$12$ band, dipping at $\sigma_w=4$ for the
heavier (Gaussian-ward) $\alpha$ where saturation is highest.

In [ ]:
# Precomputed physical mobility edge s_c = sigma_w * (cavity edge), from
# htj.jacobian_localization_phase(alpha_vals=(1.3,1.5,1.7,1.9),
#     sigma_W_vals=(1.5,2.0,2.5,3.0,3.5,4.0), P=6000).  nan = delocalised.
# To regenerate (slow, ~40 min):
# rows = htj.jacobian_localization_phase(
#     alpha_vals=(1.3,1.5,1.7,1.9), sigma_W_vals=(1.5,2.0,2.5,3.0,3.5,4.0), P=6000)
mob_alphas = np.array([1.3, 1.5, 1.7, 1.9])
mob_sigmas = np.array([1.5, 2.0, 2.5, 3.0, 3.5, 4.0])
s_c = np.array([
    [np.nan, np.nan, np.nan, np.nan],     # sigma_w=1.5  (delocalised)
    [9.69, 10.19, 10.46, 10.74],          # sigma_w=2.0
    [10.10, 10.83, 11.58, 11.90],         # sigma_w=2.5
    [10.68, 11.27, 11.43, 11.84],         # sigma_w=3.0
    [9.98, 10.96, 11.59, 10.85],          # sigma_w=3.5
    [10.32, 8.35, 6.71, 5.61],            # sigma_w=4.0
])

da = (mob_alphas[1] - mob_alphas[0]) / 2
ds = (mob_sigmas[1] - mob_sigmas[0]) / 2
mext = [mob_alphas[0] - da, mob_alphas[-1] + da, mob_sigmas[0] - ds, mob_sigmas[-1] + ds]
cmap = plt.get_cmap("plasma").copy(); cmap.set_bad("0.8")

fig, ax = plt.subplots(figsize=(7, 5.4))
im = ax.imshow(np.ma.masked_invalid(s_c), origin="lower", aspect="auto",
               extent=mext, cmap=cmap)
for i, sw in enumerate(mob_sigmas):
    for j, a in enumerate(mob_alphas):
        v = s_c[i, j]
        ax.text(a, sw, "deloc" if not np.isfinite(v) else f"{v:.1f}",
                ha="center", va="center", fontsize=8,
                color="w" if np.isfinite(v) else "0.3")
ax.axhspan(mob_sigmas[0] - ds - 1.2, mob_sigmas[0] - ds, color="0.85")
ax.text(mob_alphas.mean(), mob_sigmas[0] - ds - 0.18,
        "delocalised (unstructured / BG baseline)", ha="center", va="top",
        fontsize=8, color="0.3")
ax.set(xlabel=r"$\alpha$", ylabel=r"$\sigma_w$",
       ylim=(mob_sigmas[0] - ds - 0.45, mext[3]),
       title=r"Physical SV-localisation mobility edge $s_c$ of $J^l=D^lW^l$")
plt.colorbar(im, ax=ax, label=r"$s_c$ (singular value of $J$)")
fig.tight_layout()
fig.savefig(FIGDIR / "mobility_edge_grid.png", dpi=120)
plt.show()
print(f"physical s_c range {np.nanmin(s_c):.1f}-{np.nanmax(s_c):.1f}; "
      f"normalised s_c/sigma_w range "
      f"{np.nanmin(s_c/mob_sigmas[:,None]):.2f}-{np.nanmax(s_c/mob_sigmas[:,None]):.2f}")

### 4c. Edge-truncated propagation gain $\chi_1^{<s_c}$

The per-layer gain $\chi_1=\langle s^2\rangle$ (Poole's $\chi_1$ at $\alpha=2$)
**diverges** for the heavy-tailed Jacobian: the SV density tail
$f(s)\sim B s^{-(1+\alpha)}$ gives integrand $s^2 f\sim s^{1-\alpha}$, not
integrable for $\alpha<2$. The divergence is carried by the localised tail
$s>s_c$ -- modes that do not propagate coherent signal. The finite gain of the
extended modes is the edge-truncated $\chi_1^{<s_c}=\frac1N\sum_{s_k<s_c}s_k^2$
(sec. 6, `htj.truncated_chi1`, empirical SVD of synthetic Jacobians). Below: it is
$N$-stable and finite ($1.5$-$3.1$), whereas the full $\langle s^2\rangle$ inflates
with $N$ (right). $\chi_1^{<s_c}>1$ throughout the localised region -- the extended
modes are net-amplifying wherever a localised tail has opened; the regularised edge
of chaos $\chi_1^{<s_c}=1$ lies in the delocalised phase, where there is no $s_c$
to truncate at.

In [ ]:
# Precomputed chi_1^{<s_c} = (1/N) sum_{s<s_c} s^2 over the localised grid (s_c
# known from the cavity).  Regenerate with htj.truncated_chi1(alpha, sigma_w, s_c):
#   r = htj.truncated_chi1(1.5, 3.0, 11.27, N=400, num_matrices=40)  # -> ~2.72
ct_alphas = np.array([1.3, 1.5, 1.7, 1.9])
ct_sigmas = np.array([2.0, 2.5, 3.0, 3.5, 4.0])
chi_trunc = np.array([
    [2.80, 2.38, 1.89, 1.51],   # sigma_w=2.0
    [2.96, 2.56, 2.13, 1.75],   # 2.5
    [3.13, 2.72, 2.34, 1.99],   # 3.0
    [2.98, 2.76, 2.57, 2.21],   # 3.5
    [3.10, 2.48, 2.15, 2.17],   # 4.0
])

da = (ct_alphas[1] - ct_alphas[0]) / 2
ds = (ct_sigmas[1] - ct_sigmas[0]) / 2
cext = [ct_alphas[0] - da, ct_alphas[-1] + da, ct_sigmas[0] - ds, ct_sigmas[-1] + ds]
fig, ax = plt.subplots(figsize=(7, 5.2))
im = ax.imshow(chi_trunc, origin="lower", aspect="auto", extent=cext, cmap="viridis")
for i, sw in enumerate(ct_sigmas):
    for j, a in enumerate(ct_alphas):
        ax.text(a, sw, f"{chi_trunc[i, j]:.2f}", ha="center", va="center",
                fontsize=8, color="w")
ax.set(xlabel=r"$\alpha$", ylabel=r"$\sigma_w$",
       title=r"Edge-truncated gain $\chi_1^{<s_c}=\langle s^2\rangle_{s<s_c}$ (finite; $>1$ throughout)")
plt.colorbar(im, ax=ax, label=r"$\chi_1^{<s_c}$")
fig.tight_layout()
fig.savefig(FIGDIR / "chi1_truncated.png", dpi=120)
plt.show()
print(f"chi_1^<s_c range {chi_trunc.min():.2f}-{chi_trunc.max():.2f}; all > 1: {(chi_trunc > 1).all()}")

### 4d. Constrained-gain cutoff $c^*(\alpha,\sigma_w)$

Read the localisation boundary directly off the gain. Sort modes by their
singular-vector multifractal dimension $D_q$ (average of left/right) and form
the delocalisation-constrained gain $\chi_q(c)=\langle s^2\,|\,D_q>c\rangle$,
which decreases in $c$ as the localised heavy-$s$ tail is cut. $c^*$ is the
smallest cutoff with $\chi_q(c^*)=1$ -- the $D_q$ threshold above which the
surviving (delocalised) modes propagate at exactly unit gain. **Grey** =
ordered: even the unconstrained gain $\chi_q(0)<1$, so no cutoff reaches unit
gain. **Crimson** = chaotic-throughout: $\chi_q>1$ across the whole $c$ range.
The star marks the classical Gaussian edge of chaos $(\alpha,\sigma_w)=(2,1)$.

In [ ]:
# Precomputed constrained gain chi_q(c) = <s^2 | D_q > c> on a fine c-grid, from
# pooled SVD-with-vectors of heavy-tailed Jacobians J = D W (N=700, 20 matrices
# per (alpha, sigma_w)).  Regenerate the npz (slow, sharded SVD) with
# .agents/temp/cstar_grid.py:
#   python .agents/temp/cstar_grid.py --start 0 --end <Ncells> --workers W
#   python .agents/temp/cstar_grid.py --consolidate   # -> cstar.npz
# c* is recovered offline here by inverting chi_q(c)=1 (no SVD needed).
from matplotlib.colors import ListedColormap

_t0 = _time.time()
_d = np.load(FIGDIR / 'cstar.npz')
cs_alphas, cs_sigmas, cs_qs = _d['alphas'], _d['sigmas'], _d['qs']
cgrid, chi = _d['cgrid'], _d['chi']        # chi: (nS, nA, nQ, nC)
nS, nA, nQ, _ = chi.shape


def _invert_cstar(chi_c):
    """First c where decreasing chi(c) crosses 1 (linear interp).
    Returns (cstar, ordered, chaotic); cstar=nan where masked."""
    ok = np.isfinite(chi_c)
    if ok.sum() < 2:
        return np.nan, False, False
    c, y = cgrid[ok], chi_c[ok]
    if y[0] < 1.0:                          # unconstrained gain already < 1
        return np.nan, True, False
    below = np.where(y <= 1.0)[0]
    if below.size == 0:                     # never reaches 1 over the c range
        return np.nan, False, True
    j = below[0]
    y0, y1 = y[j - 1], y[j]
    cstar = c[j - 1] + (1.0 - y0) * (c[j] - c[j - 1]) / (y1 - y0) if y1 != y0 else c[j - 1]
    return float(cstar), False, False


cstar = np.full((nS, nA, nQ), np.nan)
ordered = np.zeros((nS, nA, nQ), bool); chaotic = np.zeros((nS, nA, nQ), bool)
for si in range(nS):
    for ai in range(nA):
        for qi in range(nQ):
            cstar[si, ai, qi], ordered[si, ai, qi], chaotic[si, ai, qi] = \
                _invert_cstar(chi[si, ai, qi])
print(f'[invert chi_q(c)=1 over {nS}x{nA}x{nQ} grid] elapsed {_time.time() - _t0:.2f}s')

_t1 = _time.time()
cs_ext = [cs_alphas[0] - 0.025, cs_alphas[-1] + 0.025,
          cs_sigmas[0] - 0.025, cs_sigmas[-1] + 0.025]
fig, axes = plt.subplots(1, nQ, figsize=(4.2 * nQ, 4.6), constrained_layout=True)
for qi, ax in enumerate(np.atleast_1d(axes)):
    im = ax.imshow(cstar[:, :, qi], origin='lower', extent=cs_ext, aspect='auto',
                   cmap='viridis', vmin=0.0, vmax=0.8)
    for mask, color in [(ordered[:, :, qi], '0.6'), (chaotic[:, :, qi], 'crimson')]:
        ax.imshow(np.where(mask, 1.0, np.nan), origin='lower', extent=cs_ext,
                  aspect='auto', cmap=ListedColormap([color]), vmin=0, vmax=1)
    ax.plot(2.0, 1.0, 'k*', ms=14)            # classical Gaussian EoC
    ax.set(title=f'$q = {cs_qs[qi]:g}$', xlabel=r'$\alpha$', ylabel=r'$\sigma_w$')
    fig.colorbar(im, ax=ax, shrink=0.85, label=r'$c^*$')
fig.suptitle(r'$c^*(\alpha,\sigma_w)=D_q$ cutoff where $\langle s^2\,|\,D_q>c\rangle=1$'
             r'   (grey=ordered $\chi(0)<1$; crimson=chaotic-throughout; $\star$=classical EoC)')
fig.savefig(FIGDIR / 'cstar_phase.png', dpi=130)
plt.show()
print(f'[plot {nQ} panels] elapsed {_time.time() - _t1:.2f}s')
print(f'c* range {np.nanmin(cstar):.2f}-{np.nanmax(cstar):.2f}; '
      f'ordered {ordered[..., 0].sum()}, chaotic {chaotic[..., 0].sum()} cells (q={cs_qs[0]:g})')

## Scope and reductions

* $\alpha = 2$ is **not** run on the heavy-tailed code path (`belinschi_quantile_scale(2, 1) = 0`); see md sec. 4. The Gaussian limit reduces to Marchenko-Pastur with parameter $\sigma_w^2 \mathbb{E}[\phi'(h)^2]$ at the fixed point.
* For unbounded $\phi$ (ReLU) the forward $q^*$ diverges and the framework does not apply; md sec. 6 / `heavy_tailed_mlp.md` sec. 4(c).
* For $\gamma \ne 1$ (non-square layers) the atom $1 - \gamma$ at zero reappears; Theorem 2 still applies with the same scalar closure (eq. 4 of the md).
* Product Jacobian across depth is deferred to a separate derivation -- it requires free-multiplicative composition of heavy-tailed S-transforms.